# 🛡️ HMO Auditor — Consola Maestra de Administración Elite

Este Notebook es el cerebro administrativo del sistema. Permite ejecutar funciones críticas fuera de la interfaz web de Streamlit, facilitando la auditoría técnica, la gestión de seguridad y la verificación de datos.

### 📑 Estructura del Notebook:
1. **Configuración del Entorno:** Carga de módulos y rutas.
2. **Gestión de Usuarios (Auth):** Auditoría de cuentas, roles y reseteo de claves.
3. **Inventario de Proyectos (Local):** Verificación de expedientes en el disco.
4. **Mantenimiento de Sistema:** Limpieza de archivos temporales y logs.

## ⚙️ 1. Configuración de Agentes y Rutas
En esta sección preparamos el sistema para que pueda leer los módulos de `HMO_Auth.py` y otras librerías internas.

In [ ]:
import os
import sys
import datetime
import json

# Aseguramos que el notebook vea los archivos de la raíz del proyecto
CURRENT_DIR = os.getcwd()
if CURRENT_DIR not in sys.path:
    sys.path.append(CURRENT_DIR)

print(f"📌 Entorno configurado en: {CURRENT_DIR}")
print(f"📅 Fecha de acceso: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 🔑 2. Gestión de Usuarios y Roles (Auditoría Forense)

El sistema de autenticación de HMO usa **SHA-256** para proteger las claves. No almacenamos contraseñas en texto plano.

### ¿Qué hace este código?
- **`get_all_users()`**: Lee el archivo `hmo_users.json` y extrae la información pública (nombre, rol, estado).
- **`load_users()`**: Módulo que asegura que el archivo de base de datos exista.
- **Lógica de Roles**: Se asignan permisos (`admin`, `auditor`, `visitante`) que Streamlit usa para ocultar o mostrar botones.

In [ ]:
from HMO_Auth import get_all_users, load_users

# Aseguramos existencia del archivo base
load_users()

def mostrar_auditoria_usuarios():
    users = get_all_users()
    print("="*70)
    print(f"{'USUARIO':<15} | {'ROL':<10} | {'ESTADO':<10} | {'ÚLTIMO ACCESO'}")
    print("="*70)
    for u in users:
        status = "🟢 ACTIVO" if u['activo'] else "🔴 INACTIVO"
        acceso = u.get('ultimo_acceso', 'Nunca') or 'Nunca'
        print(f"{u['user']:<15} | {u['rol']:<10} | {status:<10} | {acceso}")

mostrar_auditoria_usuarios()

### 🚨 RESET DE CONTRASEÑA (Uso Manual)
Si un usuario olvida su clave, puedes cambiarla aquí. Al hacerlo, el sistema activará la bandera `primer_login: True`, obligando al usuario a elegir su propia clave secreta al entrar a la app web.

In [ ]:
from HMO_Auth import hash_password, save_users, load_users

def reset_manual_password(username, new_pass):
    """
    Resetea la clave de un usuario y lo obliga a cambiarla en su próximo login.
    """
    data = load_users()
    for u in data['usuarios']:
        if u['user'].lower() == username.lower():
            u['hash'] = hash_password(new_pass)
            u['primer_login'] = True
            save_users(data)
            print(f"✅ Clave de '{username}' reseteada con éxito.")
            return True
    print(f"❌ Usuario '{username}' no encontrado.")
    return False

# Ejemplo: Descomenta la línea de abajo para resetear la clave del admin
# reset_manual_password('admin', 'HMO2024!')

## 📁 3. Inspección de Proyectos y Expedientes

Esta sección escanea la carpeta `Auditorias_HMO/` para informarnos qué empresas están siendo auditadas en este equipo.

In [ ]:
def escanear_auditorias():
    ruta_base = os.path.join(os.getcwd(), 'Auditorias_HMO')
    if not os.path.exists(ruta_base):
        print("⚠️ No se ha iniciado ninguna auditoría todavía.")
        return
    
    carpetas = [d for d in os.listdir(ruta_base) if os.path.isdir(os.path.join(ruta_base, d))]
    
    print(f"📂 Directorio Central: {ruta_base}")
    print(f"📊 Total de Empresas: {len(carpetas)}")
    print("-"*30)
    for c in carpetas:
        # Leer el archivo de estado para ver el progreso
        json_estado = os.path.join(ruta_base, c, 'audit_state.json')
        progreso = "Desconocido"
        if os.path.exists(json_estado):
            with open(json_estado, 'r') as f:
                st = json.load(f)
                progreso = f"{st.get('paso_ingesta', 0)}/5"
        
        print(f"🏢 Entidad: {c:<25} | Fase: {progreso}")

escanear_auditorias()

## 🧪 4. Sandbox de IA (Simulación de Procesamiento)
Prueba aquí cómo los agentes de HMO analizan los documentos.

In [ ]:
print("Sandbox listo para pruebas de OCR e inferencia local...")
# En futuros notebooks, incluiremos aquí las celdas de pruebas de visión por computadora.